In [2]:
import torch
import torch.nn.functional as F
import torch.nn as nn
# import torch.optim as optim
# import torch_optimizer as optim
import torch.nn.init as init

import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
import adabound
from sys import stdout
from tqdm import tqdm
from tqdm import trange
import seaborn as sns

import time
from datetime import datetime as dt

from tensorflow.keras.models import load_model

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/zorocrit/Control_Nuclear_Spins/master/C13spin/testdata.csv?token=GHSAT0AAAAAAB5HMDTB2GTBZHJM4PEGNKE2ZCHHGNA')
# df

In [4]:
# y = df[['x', 'N', 'z']]
# # y

y = df[['Xtau', 'XN']]
y

,Xtau,XN
0,2.959641,10.0
1,4.245750,7.0
2,3.594517,10.0
3,0.479873,16.0
4,2.861716,13.0
...,...,...
36744,2.910000,13.0
36745,4.258442,9.0
36746,1.675000,13.0
36747,2.935000,22.0


In [5]:
X = df[['Al', 'Ap']]
# X

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.10, random_state=100)

In [7]:
Xdata = list(np.array(X_train.values.tolist()))
print(Xdata.__len__())

33074


In [8]:
ydata = list(np.array(y_train.values.tolist()))
# print(ydata)

In [9]:
torch.manual_seed(1)

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
print(device)

cpu


In [11]:
x = torch.FloatTensor(Xdata).to(device)
y = torch.FloatTensor(ydata).to(device)

C:\Users\KIST3\AppData\Local\Temp\ipykernel_10856\2245922195.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ..\torch\csrc\utils\tensor_new.cpp:248.)
  x = torch.FloatTensor(Xdata).to(device)


In [12]:
num_data = Xdata.__len__()

num_epoch = 2000000

In [13]:
from sys import stdout


model = nn.Sequential(

    nn.Linear(2,300),

    nn.ReLU(),

    nn.Linear(300,300),

    nn.ReLU(),

    nn.Linear(300,300),

    nn.ReLU(),
    
    nn.Linear(300,300),

    nn.ReLU(),
    
    nn.Linear(300,300),

    nn.ReLU(),
    
    nn.Linear(300,300),

    nn.ReLU(),
    
    nn.Linear(300,300),

    nn.ReLU(),
    
    nn.Linear(300,300),

    nn.ReLU(),

    nn.Linear(300,2)

).to(device)

#ReLU라는 Activation Function을 사용하여, 4개의 Linear Layer로 모델 구현

# Input layer에 1개씩 데이터가 들어가므로 nn.Linear(1,2)이며, 최종적으로 1개의 값이 나와야하기에 Output Layer는 nn.Linear(4,1)



In [ ]:
 

loss_func = nn.L1Loss().to(device)

# optimizer = optim.SGD(model.parameters(),lr = 0.0002)

# optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-4, final_lr=0.05758)  #Loss: 0.0537

# optimizer = adabound.AdaBound(model.parameters(), lr=0.435 * 1e-3, final_lr=0.05558)

optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-3, final_lr=0.04978)  #Loss: 0.0537

loss_array = []

minloss = 10

for i in tqdm(range(num_epoch)) : 

    optimizer.zero_grad()

    output = model(x)

    loss = loss_func(output,y)

    loss.backward()

    optimizer.step()

    if(loss < minloss):
       minloss = loss

    loss_array.append(loss)
    if(i == 1):
      stdout.write("Minimum loss: " + str(minloss))
    if(i%100000 == 0 and i != 0):
      # print("Case "+ str(i) + ", Loss: " + str(loss.data))
      stdout.write("Minimum loss: " + str(minloss))

    if i == num_epoch - 1:

        print(loss.data)

        param_list = list(model.parameters())

        #최종 학습된 마지막 결과물의 Parameter 저장

        print(param_list)

loss_array = torch.tensor(loss_array)

loss_array.detach().numpy()

plt.plot(loss_array)

plt.show()

#Loss(y_predicted - y_real)값이 어떻게 변하는지 그래프로 도식화

In [13]:
from datetime import datetime as dt
date = dt.now()

printdate = date.strftime('%Y%m%d_%H%M%S')
print(date)

torch.save(model.state_dict(), "Xmodel_" + printdate + ".h5")

2023-04-19 17:35:55.364353


In [16]:
# model = load_model("C:/Users/KIST3/Desktop/13C/ml_04.11/Xmodel_20230419_173555.h5")
model.load_state_dict(torch.load("Xmodel_20230419_173555.h5", map_location=device))
model.eval()

Sequential(
  (0): Linear(in_features=2, out_features=300, bias=True)
  (1): ReLU()
  (2): Linear(in_features=300, out_features=300, bias=True)
  (3): ReLU()
  (4): Linear(in_features=300, out_features=300, bias=True)
  (5): ReLU()
  (6): Linear(in_features=300, out_features=300, bias=True)
  (7): ReLU()
  (8): Linear(in_features=300, out_features=300, bias=True)
  (9): ReLU()
  (10): Linear(in_features=300, out_features=300, bias=True)
  (11): ReLU()
  (12): Linear(in_features=300, out_features=300, bias=True)
  (13): ReLU()
  (14): Linear(in_features=300, out_features=300, bias=True)
  (15): ReLU()
  (16): Linear(in_features=300, out_features=2, bias=True)
)

In [17]:
X_2 = list(np.array(X.values.tolist()))
X_2_tensor = torch.FloatTensor(X_2).to(device)
y_2_input = model(X_2_tensor)
y_2_input = np.array(y_2_input.tolist())
y_2_input

array([[ 2.96878338,  9.99699306],
       [ 4.43640709,  6.98709679],
       [ 3.61910868, 10.01675129],
       ...,
       [ 1.68283594, 13.00479031],
       [ 2.9416244 , 21.95355415],
       [ 3.10761285,  6.99611664]])

In [18]:
X_input = list(np.array(X_valid.values.tolist()))
new_var =  torch.FloatTensor(X_input).to(device)
y_input = model(new_var)
y_input = np.array(y_input.tolist())
y_input

array([[ 1.4715445 ,  9.91405964],
       [ 4.16308737, 10.01140594],
       [ 1.37726974,  6.99695826],
       ...,
       [ 1.47109985,  6.47761583],
       [ 4.35567141, 22.13625145],
       [ 4.01698875,  7.42737913]])

In [19]:
y_output = np.array(y_valid.values.tolist())
y_output

array([[ 4.35176188, 10.        ],
       [ 1.91      , 10.        ],
       [ 1.36831193,  7.        ],
       ...,
       [ 1.44176854,  7.        ],
       [ 4.37756871, 22.        ],
       [ 1.57982355, 10.        ]])

In [20]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# MAE
print("X tau MAE: " + str(mean_absolute_error(y_input[:, 0], y_output[:, 0])))
print("X N MAE: " + str(mean_absolute_error(y_input[:, 1], y_output[:, 1])))
# print("Z tau MAE: " + str(mean_absolute_error(y_input[:, 2], y_output[:, 2])))
# print("Z N MAE: " + str(mean_absolute_error(y_input[:, 3], y_output[:, 3])))

print(' ')
#MSE
print("X tau MSE: " + str(mean_squared_error(y_input[:, 0], y_output[:, 0])))
print("X N MSE: " + str(mean_squared_error(y_input[:, 1], y_output[:, 1])))
# print("Z tau MSE: " + str(mean_squared_error(y_input[:, 2], y_output[:, 2])))
# print("Z N MSE: " + str(mean_squared_error(y_input[:, 3], y_output[:, 3])))

print(' ')
#r2
print("X tau r2: " + str(r2_score(y_input[:, 0], y_output[:, 0])))
print("X N r2: " + str(r2_score(y_input[:, 1], y_output[:, 1])))
# print("Z tau r2: " + str(r2_score(y_input[:, 2], y_output[:, 2])))
# print("Z N r2: " + str(r2_score(y_input[:, 3], y_output[:, 3])))

X tau MAE: 0.27118998947497486
X N MAE: 1.5229432882581437
 
X tau MSE: 0.5187856536903979
X N MSE: 16.648141847310296
 
X tau r2: 0.7098480771867661
X N r2: 0.3147041466231151


In [21]:
y2 = df[['Ztau', 'ZN']]
y2

,Ztau,ZN
0,0.176380,1.0
1,0.625249,22.0
2,1.129403,1.0
3,0.059788,1.0
4,0.197018,4.0
...,...,...
36744,1.273125,10.0
36745,2.132500,1.0
36746,0.209375,7.0
36747,0.917188,1.0


In [22]:
x2 = df[['Al', 'Ap']]
x2

,Al,Ap
0,1.977370,0.765909
1,2.673387,1.847034
2,0.572199,1.815541
3,1.159075,0.425780
4,4.427746,0.920076
...,...,...
36744,4.275027,0.802606
36745,4.104650,1.576277
36746,3.926686,0.789887
36747,2.068772,0.314327


In [23]:
x2_2 = y_2_input
x2_2 = pd.DataFrame(x2_2)
x2_2.rename(columns={0: "Xtau_pre", 1: "XN_pre"}, inplace=True)
x2_2

,Xtau_pre,XN_pre
0,2.968783,9.996993
1,4.436407,6.987097
2,3.619109,10.016751
3,0.487028,15.999538
4,2.876117,13.003598
...,...,...
36744,2.922971,13.003339
36745,4.259301,9.045465
36746,1.682836,13.004790
36747,2.941624,21.953554


In [24]:
x2_fin = pd.concat([x2, x2_2], axis = 1)
x2_fin

,Al,Ap,Xtau_pre,XN_pre
0,1.977370,0.765909,2.968783,9.996993
1,2.673387,1.847034,4.436407,6.987097
2,0.572199,1.815541,3.619109,10.016751
3,1.159075,0.425780,0.487028,15.999538
4,4.427746,0.920076,2.876117,13.003598
...,...,...,...,...
36744,4.275027,0.802606,2.922971,13.003339
36745,4.104650,1.576277,4.259301,9.045465
36746,3.926686,0.789887,1.682836,13.004790
36747,2.068772,0.314327,2.941624,21.953554


In [25]:
X_train, X_valid, y_train, y_valid = train_test_split(x2_fin, y2, test_size=0.10, random_state=100)

In [26]:
Xdata = list(np.array(X_train.values.tolist()))
Ydata = list(np.array(y_train.values.tolist()))
print(Xdata.__len__())

33074


In [27]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
print(device)

cpu


In [28]:
x22 = torch.FloatTensor(Xdata).to(device)
y22 = torch.FloatTensor(Ydata).to(device)

In [31]:
num_data = Xdata.__len__()

num_epoch = 2250000

In [33]:
model2 = nn.Sequential(

    nn.Linear(4,400),

    nn.ReLU(),

    nn.Linear(400,400),

    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,400),
    
    nn.ReLU(),

    nn.Linear(400,2)

).to(device)

In [ ]:


#ReLU라는 Activation Function을 사용하여, 4개의 Linear Layer로 모델 구현

# Input layer에 1개씩 데이터가 들어가므로 nn.Linear(1,2)이며, 최종적으로 1개의 값이 나와야하기에 Output Layer는 nn.Linear(4,1)

 

loss_func = nn.L1Loss().to(device)

# optimizer = optim.SGD(model.parameters(),lr = 0.0002)

# optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-4, final_lr=0.05758)  #Loss: 0.0537

# optimizer = adabound.AdaBound(model.parameters(), lr=0.435 * 1e-3, final_lr=0.05558)

optimizer = adabound.AdaBound(model2.parameters(), lr=0.935 * 1e-3, final_lr=0.05958)  #Loss: 0.0537

loss_array = []

minloss = 10

for i in tqdm(range(num_epoch)) : 

    optimizer.zero_grad()

    output = model2(x22)

    loss = loss_func(output,y22)

    loss.backward()

    optimizer.step()

    if(loss < minloss):
       minloss = loss

    loss_array.append(loss)
    if(i == 1):
      stdout.write("Minimum loss: " + str(minloss))
    if(i%100000 == 0 and i != 0):
      # print("Case "+ str(i) + ", Loss: " + str(loss.data))
      stdout.write("Minimum loss: " + str(minloss))

    if i == num_epoch - 1:

        print(loss.data)

        param_list = list(model.parameters())

        #최종 학습된 마지막 결과물의 Parameter 저장

        print(param_list)

loss_array = torch.tensor(loss_array)

loss_array.detach().numpy()

plt.plot(loss_array)

plt.show()

#Loss(y_predicted - y_real)값이 어떻게 변하는지 그래프로 도식화

In [30]:
from datetime import datetime as dt 
date = dt.now()

printdate = date.strftime('%Y%m%d_%H%M%S')
print(date)

torch.save(model2.state_dict(), "Zmodel2_" + printdate + ".h5")

2023-04-20 09:21:57.084097


In [35]:
# model = load_model("C:/Users/KIST3/Desktop/13C/ml_04.11/Xmodel_20230419_173555.h5")
model2.load_state_dict(torch.load("Zmodel2_20230420_092157.h5", map_location=device))
model2.eval()

Sequential(
  (0): Linear(in_features=4, out_features=400, bias=True)
  (1): ReLU()
  (2): Linear(in_features=400, out_features=400, bias=True)
  (3): ReLU()
  (4): Linear(in_features=400, out_features=400, bias=True)
  (5): ReLU()
  (6): Linear(in_features=400, out_features=400, bias=True)
  (7): ReLU()
  (8): Linear(in_features=400, out_features=400, bias=True)
  (9): ReLU()
  (10): Linear(in_features=400, out_features=400, bias=True)
  (11): ReLU()
  (12): Linear(in_features=400, out_features=400, bias=True)
  (13): ReLU()
  (14): Linear(in_features=400, out_features=400, bias=True)
  (15): ReLU()
  (16): Linear(in_features=400, out_features=2, bias=True)
)

In [36]:
X_input = list(np.array(X_valid.values.tolist()))
new_var =  torch.FloatTensor(X_input).to(device)
y_input2 = model2(new_var)
y_input2 = np.array(y_input2.tolist())
y_input2

y_output2 = np.array(y_valid.values.tolist())
y_output2

array([[0.949375  , 1.        ],
       [0.119375  , 4.        ],
       [0.12843749, 1.        ],
       ...,
       [0.09294756, 1.        ],
       [0.26633943, 4.        ],
       [0.34453125, 1.        ]])

In [37]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# MAE
print("Z tau MAE: " + str(mean_absolute_error(y_input2[:, 0], y_output2[:, 0])))
print("Z N MAE: " + str(mean_absolute_error(y_input2[:, 1], y_output2[:, 1])))
# print("Z tau MAE: " + str(mean_absolute_error(y_input[:, 2], y_output[:, 2])))
# print("Z N MAE: " + str(mean_absolute_error(y_input[:, 3], y_output[:, 3])))

print(' ')
#MSE
print("Z tau MSE: " + str(mean_squared_error(y_input2[:, 0], y_output2[:, 0])))
print("Z N MSE: " + str(mean_squared_error(y_input2[:, 1], y_output2[:, 1])))
# print("Z tau MSE: " + str(mean_squared_error(y_input[:, 2], y_output[:, 2])))
# print("Z N MSE: " + str(mean_squared_error(y_input[:, 3], y_output[:, 3])))

print(' ')
#r2
print("Z tau r2: " + str(r2_score(y_input2[:, 0], y_output2[:, 0])))
print("Z N r2: " + str(r2_score(y_input2[:, 1], y_output2[:, 1])))
# print("Z tau r2: " + str(r2_score(y_input[:, 2], y_output[:, 2])))
# print("Z N r2: " + str(r2_score(y_input[:, 3], y_output[:, 3])))

Z tau MAE: 0.32171212721389103
Z N MAE: 4.340451100294282
 
Z tau MSE: 0.24844159316021952
Z N MSE: 50.75923129975894
 
Z tau r2: -0.2971611272304189
Z N r2: -0.9079950231980933


In [38]:
# Al = 2*math.pi*0.3
# Ap = 2*math.pi*0.3

number = 16

# Al    = 2*pi * random.uniform(0.05, 0.8) #[MHz] # A_|| hyperfine term
# Ap = 2*pi* random.uniform(0.05, 0.3) #[MHz] # A_per hyperfine term

Alarr = X_valid[['Al']]
Aparr = X_valid[['Ap']]
Alarr = list(np.array(Alarr))
Aparr = list(np.array(Aparr))



Al = Alarr[number]
Ap = Aparr[number]

Al = float(Al)
Ap = float(Ap)
AAin = torch.FloatTensor([[Al, Ap]]).to(device)

AA_predict = model(AAin)

AAA_predict = np.array(AA_predict.tolist())

In [39]:
AA_predict = AA_predict.tolist()
list(np.array(AA_predict))
AA_predict[0][1]

13.004240036010742

In [40]:
AAin2 = torch.FloatTensor([Al, Ap, AA_predict[0][0], AA_predict[0][1]]).to(device)
AAin2
# AAin2
# AAXtau = AA

tensor([ 4.3986,  0.9043,  2.8826, 13.0042])

In [41]:
AA2_predict = model2(AAin2)
AA2_predict = AA2_predict.cpu()
AA2_predict

tensor([0.6127, 1.0218], grad_fn=<AddBackward0>)

In [44]:
#3/20(월)
# initialization system
# 이 코드는 13C의 nuclear spin을 초기화 시키는 코드입니다.,·
# 각 gate의 파라미터를 찾아서 초기화 시킵니다.
# initial:mixed state, target : up state
from toqito.channels import partial_trace
from qutip import *
from PIL import Image
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from scipy import linalg
import math
import matplotlib.pyplot as plt
from scipy import optimize
import random
from math import *
import pandas as pd
import time
from datetime import datetime as dt                         # 시간을 출력하기 위한 라이브러리  
import random
from mpl_toolkits.mplot3d import Axes3D
from sys import stdout
from tqdm import tqdm
from tqdm import trange
from scipy.linalg import fractional_matrix_power

totalstart = time.time() #시작 시간 저장

#Generating gate function
def UO(B1,B2,a,D1,D2):
    i   = 1j
    gamma = 2*pi*2.8
    D     = 2870
    UA = [[(B2**2+B1**2*cos(a))/(B1**2+B2**2), -i*B1*(e**(-i*D1))*sin(a)/sqrt(B1**2+B2**2), ((-1+cos(a))*B1*B2*(e**(-i*(D1-D2))))/(B1**2+B2**2)],
            [-i*B1*(e**(i*D1))*sin(a)/sqrt(B1**2+B2**2), cos(a), -i*B2*(e**(i*D2))*sin(a)/sqrt(B1**2+B2**2)],
            [((-1+cos(a))*B1*B2*e**(i*(D1-D2)))/(B1**2+B2**2), -i*B2*(e**(-i*D2))*sin(a)/sqrt(B1**2+B2**2), (B1**2+B2**2*cos(a))/(B1**2+B2**2)]]
    return UA

## Define dimension, pauli matrices
i   = 1j #1j
sx  = 1/sqrt(2)*np.array([[0, 1, 0],[1, 0, 1], [0, 1, 0]])
sy  = 1/sqrt(2)/i*np.array([[0, 1, 0], [-1, 0, 1],[0, -1, 0]])
sz  = np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]])
#sz  = [1, 0, 0; 0, -1, 0; 0, 0, 0]
I   = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])

# Rotation matrix projected into 2 level system
Sxp  = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]])
Sxm  = np.array([[0, 0, 0], [0, 0, 1], [0, 1, 0]])
Syp  = 1/i*np.array([[0, 1, 0], [-1, 0, 0], [0, 0, 0]])
Sym  = 1/i*np.array([[0, 0, 0], [0, 0, 1], [0, -1, 0]])
Szp  = np.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]])

#Gellman matrix
Sx  = np.array([[0, 0, 1],[0, 0, 0], [1, 0, 0]])
Sy  = np.array([[0, 0, -i],[0, 0, 0], [i, 0, 0]])
Sz  = np.array([[1, 0, 0],[0, 0, 0], [0, 0, -1]])

# Pauli basis for 13C nuclear spin
Ix  = 1/2*np.array([[0, 0, 1], [0, 0, 0], [1, 0, 0]])   
Iy  = 1/2/i*np.array([[0, 0, 1], [0, 0, 0], [-1, 0, 0]])
Iz  = np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]])
 

## Define sweep parameters
Sweep = 1001
N = Sweep
B = 403 #[G] magnetic field

T = 5; # sweep tau [us]
t = np.linspace(0,T,N)
n = 32; # number of pi pulses

## Define gate operations
# Single Q ms=+1
U090xp = UO(1,0,pi/4,0,0)
U090xmp = UO(1,0,-pi/4,0,0)
U090yp = UO(1,0,pi/4,pi/2,0)
U090ymp = UO(1,0,-pi/4,pi/2,0)
U180xp = UO(1,0,pi/2,0,0)
U180xmp = UO(1,0,-pi/2,0,0)

#Single Q ms=-1
U090xm = UO(0,1,pi/4,0,0)
U090xmm = UO(0,1,-pi/4,0,0)
U180xm = UO(0,1,pi/2,0,0)
U180xmm = UO(0,1,pi/2,0,0)

# Define initial state of the system

irho_p = np.array([[1,0,0],[0,0,0],[0,0,0]]) #[0,0,0;0,0,0]

irho_m = np.array([[0,0,0],[0,0,0],[0,0,1]]) #[0,0,0;0,0,1]

irho_z = np.array([[0,0,0],[0,1,0],[0,0,0]]) #[0,1,0;0,0,0]

irho_mix = np.array([[1/2,0,0],[0,1/2,0],[0,0,0]]) #[1/2,0,0;0,1/2,0;0,0,0]

irho_Z = np.array([[0,0,0],[0,0,0],[0,0,1]]) #target state

irho_MIX = np.array([[1/2,0,0],[0,0,0],[0,0,1/2]])

irho = np.kron(irho_z,irho_MIX) #initial state
trace = [1, 1, 0, 100, 100, 1000, 100] # trace of the X, Y, Z, and total density matrices
vvv = [0, 0, 0, 0] 
bbb = [0, 0, 0, 0]
normalxyz = [0, 0, 0]
    
date = dt.now()
printdate = date.strftime('%Y%m%d_%H%M%S')
print(date)

def state_fidelity(rho_1, rho_2): #fidelity
        if np.shape(rho_1) != np.shape(rho_2):
            print("Dimensions of two states do not match.")
            return 0
        else:
            sqrt_rho_1 = fractional_matrix_power(rho_1, 1 / 2)
            fidelity = np.trace(fractional_matrix_power(sqrt_rho_1 @ rho_2 @ sqrt_rho_1, 1 / 2)) ** 2
            return np.real(fidelity)

def problem(vari): 
        #for e Ry(pi/2)
        rho1 = np.kron(U090yp,I)@irho@(np.kron(U090yp,I).conj().T)                              # Ry 90도

        #for N Rx(pi/2)
        U_e2=(U_H.conj().T)@(linalg.expm(-i*E* vari[0]/2)@U_H)                                  # for tau/2
        U_e=(U_H.conj().T)@(linalg.expm(-i*E* vari[0])@U_H)                                     # for tau
        rho2=U_e2@rho1@(U_e2.conj().T)                                                          # first tau/2
        for k in range(1,2*math.trunc(vari[1])):                                                # N과 tau를 N개 생성
            rho2 = U_e@np.kron(U180xp,I) @ rho2 @ (np.kron(U180xp,I).conj().T) @ (U_e.conj().T) # N & tau
        rho3 = U_e2 @ np.kron(U180xp,I) @ rho2 @ (np.kron(U180xp,I).conj().T) @ (U_e2.conj().T) # last N & tau/2

        #for e Rx(pi/2)
        rho4 = np.kron(U090xp,I)@rho3@(np.kron(U090xp,I).conj().T)                              # Rx 90도

        #for N Rz(pi/2) //이부분이 Z pulse를 다루고 있다면 N을 따로 분리해야하나?>
        U_e2=(U_H.conj().T)@(linalg.expm(-i*E*vari[2]/2)@U_H)                                   # for tau/2
        U_e=(U_H.conj().T)@(linalg.expm(-i*E*vari[2])@U_H)                                      # for tau/2
        rho5=U_e2@rho4@(U_e2.conj().T)                                                          # first tau/2
        for k in range(1,2*math.trunc(vari[3])):                                                # N과 tau를 N개 생성
            rho5 = U_e@np.kron(U180xp,I) @ rho5 @ (np.kron(U180xp,I).conj().T) @ (U_e.conj().T) # N & tau
        rho6 = U_e2 @ np.kron(U180xp,I) @ rho5 @ (np.kron(U180xp,I).conj().T) @ (U_e2.conj().T) # last N & tau/2

        #for N Rx(pi/2)
        U_e2=(U_H.conj().T)@(linalg.expm(-i*E* vari[0]/2)@U_H)                                  # for tau/2
        U_e=(U_H.conj().T)@(linalg.expm(-i*E* vari[0])@U_H)                                     # for tau
        rho7=U_e2@rho6@(U_e2.conj().T)                                                          # first tau/2
        for k in range(1,2*math.trunc(vari[1])):                                                # N과 tau를 N개 생성
            rho7 = U_e@np.kron(U180xp,I) @ rho7 @ (np.kron(U180xp,I).conj().T) @ (U_e.conj().T) # N & tau
        rho8 = U_e2 @ np.kron(U180xp,I) @ rho7 @ (np.kron(U180xp,I).conj().T) @ (U_e2.conj().T) # last N & tau/2

        # projection&trace
        xob = (np.trace(Sxp@partial_trace(rho8,2))).real # for e spin
        yob = (np.trace(Syp@partial_trace(rho8,2))).real 
        zob = (np.trace(Szp@partial_trace(rho8,2))).real

        xx = (np.trace(Ix@partial_trace(rho8,1))).real # for N spin
        yy = (np.trace(Iy@partial_trace(rho8,1))).real
        zz = (np.trace(Iz@partial_trace(rho8,1))).real
        
        cost = 1 - state_fidelity(irho_Z, partial_trace(rho8, 1))

        cost2 = 2 * vari[0] * vari[1] + vari[2] * vari[3]
        
        if cost < 0.01:
             if (cost2<trace[5] and vari[1] % 1 == 0 and vari[3] % 1 == 0):
                trace[0] = xx
                trace[1] = yy
                trace[2] = zz
                trace[4] = cost
                trace[5] = cost2
                vvv[0] = vari[0]
                vvv[1] = vari[1]
                vvv[2] = vari[2]
                vvv[3] = vari[3]
        if(cost < trace[6] and vari[1] % 1 == 0 and vari[3] % 1 == 0):
            trace[6] = cost
            normalxyz[0] = xx
            normalxyz[1] = yy
            normalxyz[2] = zz
            
            bbb[0] = vari[0]
            bbb[1] = vari[1]
            bbb[2] = vari[2]
            bbb[3] = vari[3]
        print("cost: " + str(cost) + "x: " + str(xx) + "y: " + str(yy) + "z: " + str(zz) + "N: " + str(vari[1]) + "tau: " + str(vari[0]) + "N2: " + str(vari[3]) + "tau2: " + str(vari[2]))
        return cost
        
aa = []
cc = []
dd = []
count = 1
tot_sum = 0
tolN = 1e-15

for ccc in tqdm(range(1)): # range 번의 실험을 진행한다.
    trace = [1, 1, 0, 100, 100, 1000, 100]
    vvv = [0, 0, 0, 0]
    bbb = [0, 0, 0, 0]
    normalxyz = [0, 0, 0]
    start = time.time()
    #for making 13C nuclear random dataset
    gammaN = 2*pi*1.071e-3 #[MHz/G]
    # Al    = 2*pi * random.uniform(0.05, 0.8) #[MHz] # A_|| hyperfine term
    # Ap = 2*pi* random.uniform(0.05, 0.3) #[MHz] # A_per hyperfine term
    Al = Alarr[number]
    Ap = Aparr[number]

    #Initialization
    rho_0 = (np.kron(U090xp,I))@irho@((np.kron(U090xp,I)).conj().T) # superposition state on NV

    Sa= []

    ham = Al*np.kron(sz,Iz) + Ap*np.kron(sz,Ix) + B*gammaN*np.kron(I,Iz) # Hamiltonian
    eigvals = np.linalg.eigh(ham)[0]            # diagonalizing the Hamiltonian 여기서부터 문제 
    eigvecs = -1*np.linalg.eigh(ham)[1]         # eigenvectors
    E = np.diag(eigvals)                        # exponent of eigenvalues
    U_H= eigvecs.conj().T                       # unitary matrix formed by eigenvectors
    

    for p in range(1): # 1번의 실험을 진행한다.(지역 최적화 알고리즘을 사용할 경우에 수정한다.)
        vari=[AA_predict[0][0], AA_predict[0][1], AA2_predict[0], AA2_predict[1]]  #초기값
        # bounds = [(0.85*tau,1.15*tau),(1.0,25.0),(0.00000000000000000001*tau,0.5*tau),(1.0,25.0)] #boundary
        if(AA_predict[0][1] < mean_absolute_error(y_input[:, 1], y_output[:, 1]) + 1):
            min_AA = 1
        else:
            min_AA = AA_predict[0][1] - mean_absolute_error(y_input[:, 1], y_output[:, 1])/2
        if(AA2_predict[1] < mean_absolute_error(y_input2[:, 1], y_output2[:, 1]) + 1):
            min_AA2 = 1   
        else:
            min_AA2 = AA2_predict - mean_absolute_error(y_input2[:, 1], y_output2[:, 1])/2
        bounds = [(float(AA_predict[0][0]) * float(1 - mean_absolute_error(y_input[:, 0], y_output[:, 0])), float(AA_predict[0][1]) * float(1 + mean_absolute_error(y_input[:, 0], y_output[:, 0]))), 
                  (min_AA, float(AA_predict[0][1] + mean_absolute_error(y_input[:, 1], y_output[:, 1]))), (float(AA2_predict[0]) * float(1 - mean_absolute_error(y_input2[:, 0], y_output2[:, 0])/2), float(AA2_predict[0]) * float(1 + mean_absolute_error(y_input2[:, 0], y_output2[:, 0])/2)), 
                  (min_AA2, float(AA2_predict[1] + mean_absolute_error(y_input2[:, 1], y_output2[:, 1])/2))]
        res4 = optimize.shgo(problem,bounds=bounds,iters=5,options={'xtol':1e-4,'ftol':1e-4}) #SHGO method
        # res4 = optimize.minimize(problem, AAdata, bounds=bounds ,method='Nelder-Mead',options={'xtol':tolN,'ftol':tolN}) #Nelder-Mead method
        res4['x'][1] = round(res4['x'][1]) #rounding
        res4['x'][3] = round(res4['x'][3]) #rounding
        dd.append([Al, Ap, trace[6], bbb[0], bbb[1], bbb[2], bbb[3], normalxyz[0], normalxyz[1], normalxyz[2], res4['nfev']])
        # cc.append([Al, Ap, trace[4], vvv[0], vvv[1], vvv[2], vvv[3], trace[0], trace[1], trace[2], res4["nfev"]])
        cc.append([Al, Ap, res4['fun'], res4['x'][0], res4['x'][1], res4['x'][2], res4['x'][3]])
        # aa.append([Al, Ap, trace[6], bbb[0], bbb[1], bbb[2], bbb[3], normalxyz[0], normalxyz[1], normalxyz[2], res4['nfev']])

        # if(np.abs(res4['fun']) < 0.1 and trace[0] != 1): #fidelity가 0.05보다 작으면 성공
        #     dd.append([Al, Ap, trace[6], bbb[0], bbb[1], bbb[2], bbb[3], normalxyz[0], normalxyz[1], normalxyz[2], res4['nfev']])
        #     cc.append([Al, Ap, trace[4], vvv[0], vvv[1], vvv[2], vvv[3], trace[0], trace[1], trace[2], res4["nfev"]])
        #     end = time.time()
        #     final = end - start
        #     count = count + 1
        # else:
        #     aa.append([Al, Ap, trace[6], bbb[0], bbb[1], bbb[2], bbb[3], normalxyz[0], normalxyz[1], normalxyz[2], res4['nfev']])
        #     end = time.time()
        #     final = end - start
        #     count = count + 1


# 결과들을 list에 저장하여 csv파일로 저장


print('success')

totalend = time.time()
print(totalend - totalstart)

print(dd)
print(cc)
aa = []
cc = []
dd = []
count = 1
tot_sum = 0
tolN = 1e-15

for ccc in tqdm(range(1)): # range 번의 실험을 진행한다.
    trace = [1, 1, 0, 100, 100, 1000, 100]
    vvv = [0, 0, 0, 0]
    bbb = [0, 0, 0, 0]
    normalxyz = [0, 0, 0]
    start = time.time()
    #for making 13C nuclear random dataset
    gammaN = 2*pi*1.071e-3 #[MHz/G]
    # Al    = 2*pi * random.uniform(0.05, 0.8) #[MHz] # A_|| hyperfine term
    # Ap = 2*pi* random.uniform(0.05, 0.3) #[MHz] # A_per hyperfine term
    Al = Alarr[number]
    Ap = Aparr[number]

    #Initialization
    rho_0 = (np.kron(U090xp,I))@irho@((np.kron(U090xp,I)).conj().T) # superposition state on NV

    Sa= []

    ham = Al*np.kron(sz,Iz) + Ap*np.kron(sz,Ix) + B*gammaN*np.kron(I,Iz) # Hamiltonian
    eigvals = np.linalg.eigh(ham)[0]            # diagonalizing the Hamiltonian 여기서부터 문제 
    eigvecs = -1*np.linalg.eigh(ham)[1]         # eigenvectors
    E = np.diag(eigvals)                        # exponent of eigenvalues
    U_H= eigvecs.conj().T                       # unitary matrix formed by eigenvectors
    

    for p in range(1): # 1번의 실험을 진행한다.(지역 최적화 알고리즘을 사용할 경우에 수정한다.)
        vari=[AA_predict[0][0], AA_predict[0][1], AA2_predict[0], AA2_predict[1]]  #초기값
        # bounds = [(0.85*tau,1.15*tau),(1.0,25.0),(0.00000000000000000001*tau,0.5*tau),(1.0,25.0)] #boundary
        if(AA_predict[0][1] < mean_absolute_error(y_input[:, 1], y_output[:, 1]) + 1):
            min_AA = 1
        else:
            min_AA = AA_predict[0][1] - mean_absolute_error(y_input[:, 1], y_output[:, 1])/2
        if(AA2_predict[1] < mean_absolute_error(y_input2[:, 1], y_output2[:, 1]) + 1):
            min_AA2 = 1   
        else:
            min_AA2 = AA2_predict - mean_absolute_error(y_input2[:, 1], y_output2[:, 1])/2
        bounds = [(float(AA_predict[0][0]) * float(1 - mean_absolute_error(y_input[:, 0], y_output[:, 0])), float(AA_predict[0][1]) * float(1 + mean_absolute_error(y_input[:, 0], y_output[:, 0]))), 
                  (min_AA, float(AA_predict[0][1] + mean_absolute_error(y_input[:, 1], y_output[:, 1]))), (float(AA2_predict[0]) * float(1 - mean_absolute_error(y_input2[:, 0], y_output2[:, 0])/2), float(AA2_predict[0]) * float(1 + mean_absolute_error(y_input2[:, 0], y_output2[:, 0])/2)), 
                  (min_AA2, float(AA2_predict[1] + mean_absolute_error(y_input2[:, 1], y_output2[:, 1])/2))]
        # res4 = optimize.shgo(problem,bounds=bounds,iters=5,options={'xtol':1e-4,'ftol':1e-4}) #SHGO method
        res4 = optimize.minimize(problem, AAdata, bounds=bounds ,method='Nelder-Mead',options={'xtol':tolN,'ftol':tolN}) #Nelder-Mead method
        res4['x'][1] = round(res4['x'][1]) #rounding
        res4['x'][3] = round(res4['x'][3]) #rounding
        dd.append([Al, Ap, trace[6], bbb[0], bbb[1], bbb[2], bbb[3], normalxyz[0], normalxyz[1], normalxyz[2], res4['nfev']])
        # cc.append([Al, Ap, trace[4], vvv[0], vvv[1], vvv[2], vvv[3], trace[0], trace[1], trace[2], res4["nfev"]])
        cc.append([Al, Ap, res4['fun'], res4['x'][0], res4['x'][1], res4['x'][2], res4['x'][3]])


print('success')

totalend = time.time()
print(totalend - totalstart)

print(dd)
print(cc)

2023-04-25 10:52:04.461000


  0%|          | 0/1 [00:00<?, ?it/s]

cost: 0.5004857925161492x: 4.410823744596779e-05y: -9.024964984952026e-07z: 0.0009715850322954966N: 12.24276839188167tau: 2.100846761477364N2: 1.0tau2: 0.5141143178250699
cost: 0.5002024117303427x: -2.2378798273460336e-05y: 0.00010573890777611792z: 0.00040482346068748676N: 14.527183324268886tau: 16.530859754506544N2: 3.192072629928589tau2: 0.7112161541674229
cost: 0.4996095359077868x: -2.2013353795189957e-05y: 2.2375078992076377e-05z: -0.0007809281844244342N: 12.24276839188167tau: 16.530859754506544N2: 1.0tau2: 0.5141143178250699
cost: 0.5004757980599945x: 6.069299057927507e-05y: 3.5315754900546326e-05z: 0.0009515961199914047N: 14.527183324268886tau: 16.530859754506544N2: 1.0tau2: 0.5141143178250699
cost: 0.5001785902088502x: 9.582431161042583e-05y: -1.6124369927260872e-05z: 0.00035718041770316233N: 14.527183324268886tau: 16.530859754506544N2: 1.0tau2: 0.7112161541674229
cost: 0.5007417910562135x: 7.961695146451928e-05y: 2.325356000307862e-05z: 0.0014835821124282722N: 14.52718332426888

  0%|          | 0/1 [02:12<?, ?it/s]

cost: 0.008776957879768066x: 0.01055842098963886y: -0.08512060406886227z: -0.9824460842404668N: 13.242199924801078tau: 15.65130786182944N2: 2.7810590118169785tau2: 0.6619406950818346
cost: 0.008777129458672484x: 0.010558697226883539y: -0.08512072884140684z: -0.9824457410826546N: 13.242199924801078tau: 15.651307876730602N2: 2.7810590118169785tau2: 0.6619406950818346
cost: 0.008776957879768066x: 0.01055842098963886y: -0.08512060406886227z: -0.9824460842404668N: 13.242199939702239tau: 15.65130786182944N2: 2.7810590118169785tau2: 0.6619406950818346
cost: 0.008776980369614162x: 0.010558428080348623y: -0.08512062392577918z: -0.9824460392607746N: 13.242199924801078tau: 15.65130786182944N2: 2.7810590118169785tau2: 0.6619407099829958
cost: 0.008776957879768066x: 0.01055842098963886y: -0.08512060406886227z: -0.9824460842404668N: 13.242199924801078tau: 15.65130786182944N2: 2.7810590267181396tau2: 0.6619406950818346
cost: 0.007514020783374931x: 0.00675763006516859y: -0.08322464924331294z: -0.98497

KeyboardInterrupt: 